## **ShaRC**

In [ ]:
!wget https://sharc-data.github.io/data/sharc1-official.zip

In [ ]:
import zipfile

with zipfile.ZipFile("sharc1-official.zip", 'r') as zip_ref:
    zip_ref.extractall("sharc")

In [ ]:
import os

path = "/content/sharc/sharc1-official/json"
print(os.listdir(path))

In [ ]:
import json

file_path = "/content/sharc/sharc1-official/json/sharc_train.json"

with open(file_path, "r") as f:
    train_data = json.load(f)

print(type(train_data))
print(len(train_data))

In [ ]:
import json

print(json.dumps(train_data[0], indent=2))

In [ ]:
# --- EDA: ShARC ---
import pandas as pd
import matplotlib.pyplot as plt

records = []

for item in train_data:
    records.append({
        "utterance_id"    : item["utterance_id"],
        "tree_id"         : item["tree_id"],
        "source_url"      : item["source_url"],
        "snippet"         : item.get("snippet", ""),
        "question"        : item["question"],
        "scenario"        : item.get("scenario", ""),
        "answer"          : item["answer"],
        "history_len"     : len(item.get("history", [])),
        "evidence_len"    : len(item.get("evidence", [])),
        "has_scenario"    : len(item.get("scenario", "").strip()) > 0,
        "q_len"           : len(item["question"].split()),
        "snippet_len"     : len(item.get("snippet", "").split()),
        "scenario_len"    : len(item.get("scenario", "").split()),
    })

df = pd.DataFrame(records)

# 1. Basic counts
print("=" * 55)
print(f"Total samples              : {len(df)}")
print(f"Unique dialogue trees      : {df['tree_id'].nunique()}")
print(f"Unique source URLs         : {df['source_url'].nunique()}")

# 2. Answer distribution — direct ANSWER/ASK/ABSTAIN mapping
print(f"\n--- Answer Distribution (Decision Action Mapping) ---")
ans_counts = df["answer"].value_counts()
print(ans_counts.to_string())
print(f"\n  Yes/No      → ANSWER  : {ans_counts.get('Yes',0) + ans_counts.get('No',0)}")
print(f"  Follow-on   → ASK     : {ans_counts.get('Follow-on',0)}")
print(f"  Irrelevant  → ABSTAIN : {ans_counts.get('Irrelevant',0)}")

# 3. History depth — multi-turn context
print(f"\n--- History Depth (prior turns in dialogue) ---")
print(df["history_len"].describe().round(2).to_string())
print(f"\nSamples with no history    : {(df['history_len']==0).sum()}")
print(f"Samples with history > 0   : {(df['history_len']>0).sum()}")

# 4. Evidence/clarification turns — ASK depth
print(f"\n--- Evidence Length (clarification turns needed) ---")
print(df["evidence_len"].describe().round(2).to_string())
print(f"\nSamples needing 0 clarifications : {(df['evidence_len']==0).sum()}")
print(f"Samples needing 1+               : {(df['evidence_len']>0).sum()}")
print(f"Max clarification turns          : {df['evidence_len'].max()}")

# 5. Scenario presence — information completeness signal
print(f"\n--- Scenario Presence (user-provided context completeness) ---")
print(f"Has scenario    : {df['has_scenario'].sum()} ({100*df['has_scenario'].mean():.1f}%)")
print(f"No scenario     : {(~df['has_scenario']).sum()} ({100*(~df['has_scenario']).mean():.1f}%)")

# 6. Answer breakdown by scenario presence
print(f"\n--- Answer Type by Scenario Presence ---")
print(df.groupby(["has_scenario", "answer"]).size().unstack(fill_value=0).to_string())

# 7. Text lengths
print(f"\n--- Question Length (words) ---")
print(df["q_len"].describe().round(2).to_string())
print(f"\n--- Snippet Length (words) ---")
print(df["snippet_len"].describe().round(2).to_string())
print(f"\n--- Scenario Length (words, when present) ---")
print(df[df["has_scenario"]]["scenario_len"].describe().round(2).to_string())

# 8. Plots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("ShARC EDA — Decision-Aware Signals (Train)", fontsize=13)

# Answer distribution
colors = {"Yes": "#4CAF50", "No": "#F44336", "Follow-on": "#2196F3", "Irrelevant": "#9E9E9E"}
axes[0].bar(ans_counts.index, ans_counts.values,
            color=[colors.get(c, "#607D8B") for c in ans_counts.index])
axes[0].set_title("Answer Distribution\n(ANSWER / ASK / ABSTAIN)")
axes[0].set_ylabel("Count")

# Evidence length histogram
axes[1].hist(df["evidence_len"], bins=15, color="#FF9800", edgecolor="white")
axes[1].set_title("Clarification Turns\n(evidence length = ASK depth)")
axes[1].set_xlabel("# Clarification turns")
axes[1].set_ylabel("Frequency")

# History length histogram
axes[2].hist(df["history_len"], bins=10, color="#9C27B0", edgecolor="white")
axes[2].set_title("History Depth\n(prior dialogue turns)")
axes[2].set_xlabel("# Prior turns")
axes[2].set_ylabel("Frequency")

# Scenario presence pie
sc_counts = df["has_scenario"].value_counts()
axes[3].pie(sc_counts.values,
            labels=["Has Scenario", "No Scenario"],
            autopct="%1.1f%%", colors=["#00BCD4", "#BDBDBD"])
axes[3].set_title("Scenario Presence\n(user context completeness)")

plt.tight_layout()
plt.savefig("/content/sharc_eda.png", dpi=150)
plt.show()
print("Saved: sharc_eda.png")

In [ ]:
# --- ShARC: Sample Examples for Each EDA Case ---

import random
random.seed(42)

def show_sharc_samples(label, subset, n=3, snippet_chars=300, scenario_chars=200):
    print("\n" + "="*65)
    print(f"📌 CASE: {label}  ({len(subset)} total)")
    print("="*65)
    samples = subset.sample(min(n, len(subset)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        snippet  = row["snippet"].replace("\n", " ").strip()[:snippet_chars]
        scenario = row["scenario"].strip()[:scenario_chars] if row["has_scenario"] else "— none —"
        print(f"\n  Example {i}:")
        print(f"  SNIPPET  (regulatory text) : {snippet}...")
        print(f"  SCENARIO (user context)    : {scenario}")
        print(f"  Q        : {row['question']}")
        print(f"  ANSWER   : {row['answer']}")
        print(f"  history turns : {row['history_len']}   |   clarification turns : {row['evidence_len']}")
    print()


# ── 1. Answer = Yes → ANSWER action ──────────────────────────
show_sharc_samples(
    "ANSWER = Yes — ANSWER action",
    df[df["answer"] == "Yes"]
)

# ── 2. Answer = No → ANSWER action ───────────────────────────
show_sharc_samples(
    "ANSWER = No — ANSWER action",
    df[df["answer"] == "No"]
)

# ── 3. Answer = Follow-on → ASK action ───────────────────────
show_sharc_samples(
    "ANSWER = Follow-on — ASK action (clarification needed)",
    df[df["answer"] == "Follow-on"]
)

# ── 4. Answer = Irrelevant → ABSTAIN action ───────────────────
show_sharc_samples(
    "ANSWER = Irrelevant — ABSTAIN action",
    df[df["answer"] == "Irrelevant"]
)

# ── 5. No scenario provided — incomplete information state ────
show_sharc_samples(
    "NO SCENARIO — incomplete user context",
    df[~df["has_scenario"]]
)

# ── 6. Has scenario + Follow-on — info given but still needs ASK
show_sharc_samples(
    "HAS SCENARIO + Follow-on — context given but still incomplete",
    df[(df["has_scenario"]) & (df["answer"] == "Follow-on")]
)

# ── 7. Max clarification depth — full evidence chain ─────────
print("\n" + "="*65)
print("📌 CASE: MAX CLARIFICATION DEPTH — full evidence chain")
print("="*65)

max_ev_idx = df["evidence_len"].idxmax()
max_row    = df.loc[max_ev_idx]

# Fetch raw item for evidence detail
raw_item = next(x for x in train_data if x["utterance_id"] == max_row["utterance_id"])
snippet  = max_row["snippet"].replace("\n", " ").strip()[:400]
scenario = max_row["scenario"].strip()[:200] if max_row["has_scenario"] else "— none —"

print(f"\n  SNIPPET  : {snippet}...")
print(f"  SCENARIO : {scenario}")
print(f"  Q        : {max_row['question']}")
print(f"  ANSWER   : {max_row['answer']}")
print(f"  Clarification chain ({max_row['evidence_len']} turns):")
for j, ev in enumerate(raw_item["evidence"], 1):
    print(f"    Turn {j}: Q: {ev['follow_up_question']}  →  A: {ev['follow_up_answer']}")

# ── 8. Multi-turn history example ────────────────────────────
print("\n" + "="*65)
print("📌 CASE: MAX HISTORY DEPTH — longest prior dialogue")
print("="*65)

max_hist_idx = df["history_len"].idxmax()
max_hist_row = df.loc[max_hist_idx]
raw_hist_item = next(x for x in train_data if x["utterance_id"] == max_hist_row["utterance_id"])

snippet  = max_hist_row["snippet"].replace("\n", " ").strip()[:400]
scenario = max_hist_row["scenario"].strip()[:200] if max_hist_row["has_scenario"] else "— none —"

print(f"\n  SNIPPET  : {snippet}...")
print(f"  SCENARIO : {scenario}")
print(f"  Q        : {max_hist_row['question']}")
print(f"  ANSWER   : {max_hist_row['answer']}")
print(f"  History ({max_hist_row['history_len']} prior turns):")
for j, h in enumerate(raw_hist_item["history"], 1):
    print(f"    Turn {j}: Q: {h['follow_up_question']}  →  A: {h['follow_up_answer']}")

## **Contract-NLI**

In [ ]:
!wget https://github.com/stanfordnlp/contract-nli/raw/gh-pages/resources/contract-nli.zip -O contract_nli.zip

In [ ]:
import zipfile

with zipfile.ZipFile("contract_nli.zip", 'r') as zip_ref:
    zip_ref.extractall("contract_nli")

print("Unzipped successfully")

In [ ]:
import json

file_path = "/content/contract_nli/contract-nli/train.json"

with open(file_path, "r") as f:
    data = json.load(f)

print(type(data))  # should be dict

In [ ]:
print(json.dumps(data, indent=2)[:4000])

In [ ]:
print(json.dumps({
    "documents": data["documents"][:1],  # only 1 doc
    "labels": data["labels"]
}, indent=2))

In [ ]:
print(json.dumps(data["documents"][0], indent=2))

In [ ]:
print(json.dumps(data["labels"], indent=2))

In [ ]:
# --- EDA: ContractNLI ---
import pandas as pd
import matplotlib.pyplot as plt

documents = data["documents"]
labels    = data["labels"]

records = []

for doc in documents:
    doc_text = doc["text"]
    char_spans = doc["spans"]          # list of [start, end] char offsets

    for annot_set in doc["annotation_sets"]:
        for label_id, annot in annot_set["annotations"].items():
            choice     = annot["choice"]
            span_idxs  = annot["spans"]  # indices into char_spans list

            # Resolve actual text for each supporting span
            span_texts = []
            for idx in span_idxs:
                if idx < len(char_spans):
                    s, e = char_spans[idx]
                    span_texts.append(doc_text[s:e].strip())

            records.append({
                "doc_id"        : doc["id"],
                "file_name" : doc["file_name"],
                "label_id"      : label_id,
                "short_desc"    : labels[label_id]["short_description"],
                "hypothesis"    : labels[label_id]["hypothesis"],
                "choice"        : choice,
                "num_spans"     : len(span_idxs),
                "doc_len"       : len(doc_text.split()),
                "span_texts"    : span_texts,
                "doc_text"      : doc_text,
            })

df = pd.DataFrame(records)

# 1. Basic counts
print("=" * 50)
print(f"Total documents       : {len(documents)}")
print(f"Total annotations     : {len(df)}")
print(f"Unique NDA labels     : {df['label_id'].nunique()}")
print(f"Avg annotations/doc   : {len(df)/len(documents):.2f}")

# 2. Choice distribution — core ANSWER/ABSTAIN signal
print(f"\n--- Choice Distribution (Decision Action Signal) ---")
choice_counts = df["choice"].value_counts()
print(choice_counts.to_string())
print(f"\nNotMentioned rate (ABSTAIN signal) : {100*choice_counts.get('NotMentioned',0)/len(df):.1f}%")
print(f"Entailment rate   (ANSWER=yes)     : {100*choice_counts.get('Entailment',0)/len(df):.1f}%")
print(f"Contradiction rate (ANSWER=no)     : {100*choice_counts.get('Contradiction',0)/len(df):.1f}%")

# 3. Per-label breakdown — which clauses are most often absent
print(f"\n--- Per-Label NotMentioned Rate (missing info per clause) ---")
label_pivot = df.groupby(["short_desc", "choice"]).size().unstack(fill_value=0)
label_pivot["NotMentioned_%"] = (
    label_pivot.get("NotMentioned", 0) /
    label_pivot.sum(axis=1) * 100
).round(1)
print(label_pivot.sort_values("NotMentioned_%", ascending=False).to_string())

# 4. Supporting span count — proxy for evidence sufficiency
print(f"\n--- Supporting Spans per Annotation (evidence load) ---")
print(df[df["choice"] != "NotMentioned"]["num_spans"].describe().round(2).to_string())

# 5. Document length
print(f"\n--- Document Length (words) ---")
doc_lens = df.drop_duplicates("doc_id")["doc_len"]
print(doc_lens.describe().round(2).to_string())

# 6. Plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("ContractNLI EDA — Decision-Aware Signals", fontsize=13)

# Choice pie
colors = {"Entailment": "#4CAF50", "Contradiction": "#F44336", "NotMentioned": "#9E9E9E"}
axes[0].pie(
    choice_counts.values,
    labels=choice_counts.index,
    autopct="%1.1f%%",
    colors=[colors.get(c, "#2196F3") for c in choice_counts.index]
)
axes[0].set_title("Choice Distribution\n(ANSWER / ABSTAIN signal)")

# Per-label NotMentioned rate
nm_rates = label_pivot["NotMentioned_%"].sort_values(ascending=True)
axes[1].barh(nm_rates.index, nm_rates.values, color="#607D8B")
axes[1].set_title("NotMentioned % per Clause\n(ABSTAIN signal by label)")
axes[1].set_xlabel("NotMentioned %")
axes[1].tick_params(axis='y', labelsize=7)

# Doc length histogram
axes[2].hist(doc_lens, bins=20, color="#795548", edgecolor="white")
axes[2].set_title("Document Length\n(words per contract)")
axes[2].set_xlabel("Word count")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("/content/contractnli_eda.png", dpi=150)
plt.show()
print("Saved: contractnli_eda.png")

In [ ]:
# --- ContractNLI: Sample Examples for Each EDA Case ---

import random
random.seed(42)

def show_contractnli_samples(label, subset_df, n=3, context_chars=400):
    print("\n" + "="*65)
    print(f"📌 CASE: {label}  ({len(subset_df)} total)")
    print("="*65)
    samples = subset_df.sample(min(n, len(subset_df)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        # If spans exist use them as context, else fallback to doc start
        ctx = row["doc_text"].replace("\n", " ").strip()[:context_chars]
        # if row["span_texts"]:
        #     ctx = " | ".join(row["span_texts"])[:context_chars]
        # else:
        #     ctx = row["doc_text"].replace("\n", " ").strip()[:context_chars]

        print(f"\n  Example {i}:")
        print(f"  DOC ID    : {row['doc_id']}   |   FILE: {row['file_name']}")
        print(f"  CLAUSE    : {row['short_desc']} ({row['label_id']})")
        print(f"  HYPOTHESIS: {row['hypothesis']}")
        print(f"  CONTEXT   : {ctx}...")
        print(f"  CHOICE    : {row['choice']}   |   supporting spans: {row['num_spans']}")
    print()


# ── 1. Entailment — ANSWER = yes ─────────────────────────────
show_contractnli_samples(
    "ENTAILMENT — ANSWER action (clause supported)",
    df[df["choice"] == "Entailment"]
)

# ── 2. Contradiction — ANSWER = no ───────────────────────────
show_contractnli_samples(
    "CONTRADICTION — ANSWER action (clause contradicted)",
    df[df["choice"] == "Contradiction"]
)

# ── 3. NotMentioned — ABSTAIN action ─────────────────────────
show_contractnli_samples(
    "NOT MENTIONED — ABSTAIN action (clause absent from contract)",
    df[df["choice"] == "NotMentioned"]
)

# ── 4. Highest NotMentioned clause (most often absent) ────────
top_absent_label = (
    df[df["choice"] == "NotMentioned"]["label_id"]
    .value_counts().idxmax()
)
show_contractnli_samples(
    f"MOST ABSENT CLAUSE: {top_absent_label} — {labels[top_absent_label]['short_description']}",
    df[(df["label_id"] == top_absent_label) & (df["choice"] == "NotMentioned")]
)

# ── 5. Highest evidence load (most supporting spans) ──────────
print("\n" + "="*65)
print("📌 CASE: MAX EVIDENCE LOAD — annotation with most supporting spans")
print("="*65)
max_row = df.loc[df["num_spans"].idxmax()]
ctx = " | ".join(max_row["span_texts"])[:500]
print(f"\n  CLAUSE    : {max_row['short_desc']} ({max_row['label_id']})")
print(f"  HYPOTHESIS: {max_row['hypothesis']}")
print(f"  CONTEXT   : {ctx}...")
print(f"  CHOICE    : {max_row['choice']}   |   supporting spans: {max_row['num_spans']}")

# ── 6. Zero-span Entailment — interesting edge case ───────────
zero_span_entail = df[(df["choice"] == "Entailment") & (df["num_spans"] == 0)]
if len(zero_span_entail) > 0:
    show_contractnli_samples(
        "EDGE CASE: Entailment with 0 supporting spans",
        zero_span_entail
    )

## **QUAC**

In [ ]:
!wget https://s3.amazonaws.com/my89public/quac/train_v0.2.json
!wget https://s3.amazonaws.com/my89public/quac/val_v0.2.json

In [ ]:
import json

with open("train_v0.2.json") as f:
    train_data = json.load(f)

with open("val_v0.2.json") as f:
    val_data = json.load(f)

print(type(train_data))

In [ ]:
sample = train_data["data"][0]

import json
print(json.dumps(sample, indent=2))

In [ ]:
para = sample["paragraphs"][0]

context = para["context"]
qas = para["qas"]

print("CONTEXT:\n")
print(context[:500])

print("\n--- Q&A ---\n")

for qa in qas[:5]:
    print("Q:", qa["question"])
    print("A:", qa["answers"][0]["text"])
    print("YES/NO:", qa["yesno"])
    print("FOLLOW-UP:", qa["followup"])
    print("-"*50)

In [ ]:
# --- EDA: QuAC ---
import pandas as pd
import matplotlib.pyplot as plt

def extract_quac_records(raw):
    records = []
    turn_depths = []
    for article in raw["data"]:
        for para in article["paragraphs"]:
            qas = para["qas"]
            turn_depths.append(len(qas))
            for qa in qas:
                ans_text = qa["answers"][0]["text"] if qa["answers"] else "CANNOTANSWER"
                records.append({
                    "question": qa["question"],
                    "answer_text": ans_text,
                    "followup": qa.get("followup", "n"),
                    "yesno": qa.get("yesno", "x"),
                    "q_len": len(qa["question"].split()),
                    "a_len": len(ans_text.split()) if ans_text != "CANNOTANSWER" else 0,
                })
    return pd.DataFrame(records), turn_depths

train_df, train_turns = extract_quac_records(train_data)
val_df, val_turns = extract_quac_records(val_data)

for split_name, df, turns in [("TRAIN", train_df, train_turns), ("VAL", val_df, val_turns)]:
    cannotanswer = (df["answer_text"] == "CANNOTANSWER").sum()
    answerable = len(df) - cannotanswer

    print("=" * 50)
    print(f"SPLIT: {split_name}")
    print(f"Total QA pairs        : {len(df)}")
    print(f"Total dialogues       : {len(turns)}")
    print(f"Avg turns/dialogue    : {sum(turns)/len(turns):.2f}")

    print(f"\n--- Answerability ---")
    print(f"Answerable            : {answerable} ({100*answerable/len(df):.1f}%)")
    print(f"CANNOTANSWER          : {cannotanswer} ({100*cannotanswer/len(df):.1f}%)")

    print(f"\n--- Followup Distribution (ASK signal) ---")
    print(df["followup"].value_counts().to_string())

    print(f"\n--- Yes/No Flag Distribution ---")
    print(df["yesno"].value_counts().to_string())

    print(f"\n--- Question Length (words) ---")
    print(df["q_len"].describe().round(2).to_string())

# Plot for train
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("QuAC EDA — Decision-Aware Signals (Train)", fontsize=13)

cannotanswer_c = (train_df["answer_text"] == "CANNOTANSWER").sum()
answerable_c = len(train_df) - cannotanswer_c

axes[0].pie([answerable_c, cannotanswer_c],
            labels=["Answerable", "CANNOTANSWER"],
            autopct="%1.1f%%", colors=["#4CAF50", "#F44336"])
axes[0].set_title("Answerability\n(ANSWER vs ABSTAIN)")

fu = train_df["followup"].value_counts()
axes[1].bar(fu.index, fu.values, color=["#2196F3", "#FF9800", "#9C27B0"])
axes[1].set_title("Followup Flag\n(ASK signal)")
axes[1].set_xlabel("Followup Flag")
axes[1].set_ylabel("Count")

yn = train_df["yesno"].value_counts()
axes[2].bar(yn.index, yn.values, color=["#00BCD4", "#FF5722", "#8BC34A"])
axes[2].set_title("Yes/No Distribution\n(Answer type)")
axes[2].set_xlabel("yesno Flag")

axes[3].hist(train_turns, bins=20, color="#607D8B", edgecolor="white")
axes[3].set_title("Multi-turn Depth\nper Dialogue")
axes[3].set_xlabel("# Turns")
axes[3].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("/content/quac_eda.png", dpi=150)
plt.show()
print("Saved: quac_eda.png")

In [ ]:

# --- QuAC: Sample Examples for Each EDA Case ---

import random
import pandas as pd
random.seed(42)

# Rebuild rich records WITH context
all_qas_rich = []
turn_depth_map = {}

for article in train_data["data"]:
    for para in article["paragraphs"]:
        context = para.get("context", "")
        depth = len(para["qas"])
        if depth not in turn_depth_map:
            turn_depth_map[depth] = {"context": context, "qas": para["qas"]}

        for qa in para["qas"]:
            ans_text = qa["answers"][0]["text"] if qa["answers"] else "CANNOTANSWER"
            all_qas_rich.append({
                "context"        : context,
                "question"       : qa["question"],
                "answer_text"    : ans_text,
                "followup"       : qa.get("followup", "n"),
                "yesno"          : qa.get("yesno", "x"),
                "is_cannotanswer": ans_text == "CANNOTANSWER",
            })

df_rich = pd.DataFrame(all_qas_rich)


def show_samples(label, subset_df, n=3, context_chars=300):
    print("\n" + "="*65)
    print(f"📌 CASE: {label}  ({len(subset_df)} total)")
    print("="*65)
    samples = subset_df.sample(min(n, len(subset_df)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        ctx_preview = row["context"].replace("\n", " ").strip()[:context_chars]
        print(f"\n  Example {i}:")
        print(f"  CONTEXT  : {ctx_preview}...")
        print(f"  Q        : {row['question']}")
        print(f"  A        : {row['answer_text']}")
        print(f"  followup : {row['followup']}   |   yesno: {row['yesno']}")
    print()


# ── 1. Answerable (ANSWER action) ────────────────────────────
show_samples(
    "ANSWERABLE — ANSWER action",
    df_rich[~df_rich["is_cannotanswer"]]
)

# ── 2. CANNOTANSWER (ABSTAIN action) ─────────────────────────
show_samples(
    "CANNOTANSWER — ABSTAIN action",
    df_rich[df_rich["is_cannotanswer"]]
)

# ── 3. Followup Needed → y  (ASK signal) ─────────────────────
show_samples(
    "FOLLOWUP = y  — ASK signal",
    df_rich[df_rich["followup"] == "y"]
)

# ── 4. No Followup → n  (terminal turn) ──────────────────────
show_samples(
    "FOLLOWUP = n  — no further clarification needed",
    df_rich[df_rich["followup"] == "n"]
)

# ── 5. Maybe followup → m  (uncertain) ───────────────────────
if "m" in df_rich["followup"].values:
    show_samples(
        "FOLLOWUP = m  — maybe (ambiguous signal)",
        df_rich[df_rich["followup"] == "m"]
    )

# ── 6. Yes/No flagged answers ────────────────────────────────
show_samples(
    "YESNO = y  — yes/no type answer",
    df_rich[df_rich["yesno"] == "y"]
)

# ── 7. Multi-turn depth — full dialogue with context ─────────
print("\n" + "="*65)
print("📌 CASE: MULTI-TURN DEPTH — full dialogue examples")
print("="*65)

all_depths = sorted(turn_depth_map.keys())
min_depth  = min(all_depths)
max_depth  = max(all_depths)

for label, target_depth in [
    (f"MIN depth ({min_depth} turns)", min_depth),
    (f"MAX depth ({max_depth} turns)", max_depth)
]:
    entry = turn_depth_map.get(target_depth)
    if entry:
        ctx_preview = entry["context"].replace("\n", " ").strip()[:400]
        print(f"\n  ── {label} ──")
        print(f"  CONTEXT : {ctx_preview}...")
        for i, qa in enumerate(entry["qas"], 1):
            ans = qa["answers"][0]["text"] if qa["answers"] else "CANNOTANSWER"
            print(f"\n  Turn {i}:")
            print(f"    Q : {qa['question']}")
            print(f"    A : {ans}")
            print(f"    followup={qa.get('followup','?')}  yesno={qa.get('yesno','?')}")

## **HotPotQA**

In [ ]:
from datasets import load_dataset # Load HotpotQA (full wiki setting is standard)
dataset = load_dataset("hotpot_qa", "fullwiki") # Access splits
train_data = dataset["train"]
val_data = dataset["validation"]
print(train_data[0])

In [ ]:
import json

sample = train_data[0]

print(json.dumps(sample, indent=4))

In [ ]:
# --- EDA: HotpotQA ---
import pandas as pd
import matplotlib.pyplot as plt

def extract_hotpot_records(split):
    records = []
    for item in split:
        records.append({
            "question": item["question"],
            "answer": item["answer"],
            "type": item["type"],
            "level": item["level"],
            "q_len": len(item["question"].split()),
            "a_len": len(item["answer"].split()),
            "num_supporting_facts": len(item["supporting_facts"]["title"]),
            "num_context_docs": len(item["context"]["title"]),
        })
    return pd.DataFrame(records)

train_df = extract_hotpot_records(train_data)
val_df = extract_hotpot_records(val_data)

for split_name, df in [("TRAIN", train_df), ("VAL", val_df)]:
    print("=" * 50)
    print(f"SPLIT: {split_name}")
    print(f"Total samples         : {len(df)}")

    print(f"\n--- Question Type (reasoning demand) ---")
    print(df["type"].value_counts().to_string())

    print(f"\n--- Difficulty Level ---")
    print(df["level"].value_counts().to_string())

    print(f"\n--- Question Length (words) ---")
    print(df["q_len"].describe().round(2).to_string())

    print(f"\n--- Answer Length (words) ---")
    print(df["a_len"].describe().round(2).to_string())

    # Noteworthy: supporting facts needed — proxy for information sufficiency requirement
    print(f"\n--- Supporting Facts Required (multi-hop evidence load) ---")
    print(df["num_supporting_facts"].value_counts().sort_index().to_string())

# Plot for train
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("HotpotQA EDA — Decision-Aware Signals (Train)", fontsize=13)

type_counts = train_df["type"].value_counts()
axes[0].bar(type_counts.index, type_counts.values, color=["#3F51B5", "#E91E63"])
axes[0].set_title("Question Type\n(Reasoning demand)")
axes[0].set_ylabel("Count")

level_counts = train_df["level"].value_counts()
axes[1].bar(level_counts.index, level_counts.values, color=["#4CAF50", "#FF9800", "#F44336"])
axes[1].set_title("Difficulty Level\n(Complexity signal)")
axes[1].set_ylabel("Count")

axes[2].hist(train_df["q_len"], bins=25, color="#009688", edgecolor="white")
axes[2].set_title("Question Length\n(words)")
axes[2].set_xlabel("Word count")
axes[2].set_ylabel("Frequency")

axes[3].hist(train_df["num_supporting_facts"], bins=10, color="#795548", edgecolor="white")
axes[3].set_title("Supporting Facts Required\n(Information load per query)")
axes[3].set_xlabel("# Supporting facts")
axes[3].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("/content/hotpotqa_eda.png", dpi=150)
plt.show()
print("Saved: hotpotqa_eda.png")

In [ ]:
# --- HotpotQA: Sample Examples for Each EDA Case ---

import random
import pandas as pd
random.seed(42)

# Rebuild rich records
all_records = []

for item in train_data:
    # Build a readable context from supporting facts only (relevant sentences)
    sf_titles  = item["supporting_facts"]["title"]
    sf_sentids = item["supporting_facts"]["sent_id"]
    ctx_titles = item["context"]["title"]
    ctx_sents  = item["context"]["sentences"]

    # Map title -> sentences for quick lookup
    title_to_sents = {t: s for t, s in zip(ctx_titles, ctx_sents)}

    supporting_text = []
    for t, sid in zip(sf_titles, sf_sentids):
        if t in title_to_sents:
            sents = title_to_sents[t]
            if sid < len(sents):
                supporting_text.append(f"[{t}] {sents[sid]}")
    context_preview = " | ".join(supporting_text)

    all_records.append({
        "question"          : item["question"],
        "answer"            : item["answer"],
        "type"              : item["type"],
        "level"             : item["level"],
        "num_supporting"    : len(sf_titles),
        "context_preview"   : context_preview,
    })

df_rich = pd.DataFrame(all_records)


def show_samples(label, subset_df, n=3, context_chars=400):
    print("\n" + "="*65)
    print(f"📌 CASE: {label}  ({len(subset_df)} total)")
    print("="*65)
    samples = subset_df.sample(min(n, len(subset_df)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        ctx = row["context_preview"].strip()[:context_chars]
        print(f"\n  Example {i}:")
        print(f"  CONTEXT (supporting facts) : {ctx}...")
        print(f"  Q       : {row['question']}")
        print(f"  A       : {row['answer']}")
        print(f"  type    : {row['type']}   |   level: {row['level']}   |   supporting facts: {row['num_supporting']}")
    print()


# ── 1. Question Type: comparison ─────────────────────────────
show_samples(
    "TYPE = comparison",
    df_rich[df_rich["type"] == "comparison"]
)

# ── 2. Question Type: bridge ──────────────────────────────────
show_samples(
    "TYPE = bridge  (multi-hop reasoning)",
    df_rich[df_rich["type"] == "bridge"]
)

# ── 3. Difficulty: easy ───────────────────────────────────────
show_samples(
    "LEVEL = easy",
    df_rich[df_rich["level"] == "easy"]
)

# ── 4. Difficulty: medium ─────────────────────────────────────
show_samples(
    "LEVEL = medium",
    df_rich[df_rich["level"] == "medium"]
)

# ── 5. Difficulty: hard ───────────────────────────────────────
show_samples(
    "LEVEL = hard",
    df_rich[df_rich["level"] == "hard"]
)

# ── 6. Supporting facts load — min and max ────────────────────
print("\n" + "="*65)
print("📌 CASE: SUPPORTING FACTS LOAD — min and max examples")
print("="*65)

min_sf = df_rich["num_supporting"].min()
max_sf = df_rich["num_supporting"].max()

for label, val in [(f"MIN supporting facts ({min_sf})", min_sf),
                   (f"MAX supporting facts ({max_sf})", max_sf)]:
    subset = df_rich[df_rich["num_supporting"] == val]
    row    = subset.sample(1, random_state=42).iloc[0]
    ctx    = row["context_preview"].strip()[:500]
    print(f"\n  ── {label} ──")
    print(f"  CONTEXT (supporting facts) : {ctx}...")
    print(f"  Q       : {row['question']}")
    print(f"  A       : {row['answer']}")
    print(f"  type    : {row['type']}   |   level: {row['level']}")